## 1. Loading the structured dataset

In the previous notebook, we transformed raw character-level screenplay files into a structured dataset stored in the `data/interim/` directory.

In this notebook, we will use that dataset as the starting point for preprocessing and feature engineering.

## Why this matters

Working from an intermediate dataset ensures:

- reproducibility of the pipeline
- separation between raw data and processed data
- faster iteration for analytical tasks

## Expected output

We expect to load a DataFrame containing all character-level text entries with their associated metadata.

In [ ]:
from pathlib import Path
import sys

# Add project root to path to find src module
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
from src.utils.paths import PROJECT_ROOT

interim_path = PROJECT_ROOT / "data" / "interim" / "character_lines.parquet"

df = pd.read_parquet(interim_path)

print("Shape:", df.shape)
df.head()

## 2. Text normalization and cleaning

Before performing any aggregation or NLP analysis, it is necessary to clean and normalize the text data.

The goal of this step is not to aggressively transform the text, but rather to ensure consistency and remove noise that may affect downstream analysis.

## Why this matters

Raw screenplay text often contains:

- encoding artifacts (e.g., curly quotes, OCR issues)
- inconsistent punctuation
- non-standard characters

Cleaning the text improves the reliability of:

- emotion analysis
- text-based feature extraction
- aggregation consistency

## Cleaning strategy

We apply a **minimal and controlled cleaning approach**, including:

- normalization of quotation marks
- removal of non-standard characters
- lowercasing for consistency

We intentionally avoid heavy transformations such as stemming or stopword removal, as they may distort narrative meaning.

In [6]:
import re

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalize quotes
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')

    # Fix common OCR issues in contractions
    text = re.sub(r"'1(?=[a-zA-Z])", "'", text)   # e.g., you'1ll -> you'll
    text = re.sub(r"(?<=[a-zA-Z])1(?=[a-zA-Z])", "", text)  # generic in-word OCR noise

    # Remove strange characters but keep basic punctuation
    text = re.sub(r"[^a-zA-Z0-9\s\.\,\!\?\']", " ", text)

    # Lowercase
    text = text.lower()

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text"]].head(10)

,text,clean_text
0,Michelle's eyes slowly open. She's back on the...,michelle's eyes slowly open. she's back on the...
1,You should eat the eggs. You’1ll be dreaming o...,you should eat the eggs. you'll be dreaming of...
2,"In a couple of weeks, everything's going to be...","in a couple of weeks, everything's going to be..."
3,"You've got no idea what happened, do you? Mich...","you've got no idea what happened, do you? mich..."
4,There was an attack last night. Some kind of b...,there was an attack last night. some kind of b...
5,I was rushing back here when I came across you...,i was rushing back here when i came across you...
6,I think you're smart enough to know that if I ...,i think you're smart enough to know that if i ...
7,I’m a doctor. I had to take off your jeans to ...,i'm a doctor. i had to take off your jeans to ...
8,You can’t leave. The entire countryside is bla...,you can't leave. the entire countryside is bla...
9,"You go outside, you die. But this place has a ...","you go outside, you die. but this place has a ..."


The text normalization step produced cleaner and more consistent textual data while preserving the narrative content of the screenplay lines.

The updated cleaning function successfully corrected common formatting and OCR-related issues, such as:

- curly quotation marks
- inconsistent apostrophes
- character-level artifacts embedded inside words

Importantly, the cleaning strategy remains intentionally conservative. The text is still readable and semantically faithful to the original screenplay content, which is essential for downstream tasks such as emotion analysis and character-level interpretation.

At this stage, the dataset now contains both:

- the original raw text
- a normalized version (`clean_text`) ready for analytical use

## 3. Building scene-level analytical units

The dataset is currently organized at the level of individual character text entries. While this is useful for fine-grained text analysis, many downstream tasks require a higher-level representation.

In this section, we construct scene-level analytical units using the combination of:

- `movie_id`
- `segment_id`
- `scene_id`

This `(segment_id, scene_id)` pair serves as the project's operational approximation of a scene.

## Why this matters

Scene-level aggregation is necessary for tasks such as:

- identifying which characters co-occur in the same narrative unit
- building character interaction graphs
- analyzing scene-level emotional context
- deriving structural features from the screenplay

## Expected output

We expect to obtain a new dataset where each row represents a scene approximation, including:

- the list of characters present
- the number of characters in the scene
- the number of text entries in the scene
- aggregated scene text

In [9]:
scene_df = (
    df.groupby(["movie_id", "segment_id", "scene_id", "scene_key"])
      .agg(
          characters_present=("character_name", lambda x: sorted(set(x))),
          n_characters=("character_name", "nunique"),
          n_lines=("character_name", "size"),
          scene_text=("clean_text", " ".join)
      )
      .reset_index()
)

print("Scene-level dataset shape:", scene_df.shape)
scene_df.head()

Scene-level dataset shape: (1293787, 8)


,movie_id,segment_id,scene_id,scene_key,characters_present,n_characters,n_lines,scene_text
0,10 Cloverfield Lane_1179933,0,0,0_0,"[Howard, Michelle]",2,2,michelle's eyes slowly open. she's back on the...
1,10 Cloverfield Lane_1179933,0,1,0_1,[Howard],1,1,you should eat the eggs. you'll be dreaming of...
2,10 Cloverfield Lane_1179933,0,2,0_2,[Howard],1,1,"in a couple of weeks, everything's going to be..."
3,10 Cloverfield Lane_1179933,0,3,0_3,[Michelle],1,1,i called the police. they'll be here any minut...
4,10 Cloverfield Lane_1179933,0,4,0_4,[Howard],1,1,"you've got no idea what happened, do you? mich..."


### Revisiting the definition of a scene

Initial aggregation using `(segment_id, scene_id)` resulted in an extremely high number of scene units, suggesting that `scene_id` represents a very fine-grained segmentation of the text rather than meaningful narrative scenes.

To construct more useful analytical units, we redefine the operational notion of a scene using only:

- `movie_id`
- `segment_id`

This approach aggregates multiple micro-segments into a broader narrative unit, allowing for:

- meaningful character co-occurrence
- richer scene context
- more reliable interaction modeling

This adjustment is critical for ensuring that downstream graph construction and analysis are valid.

In [10]:
scene_df = (
    df.groupby(["movie_id", "segment_id"])
      .agg(
          characters_present=("character_name", lambda x: sorted(set(x))),
          n_characters=("character_name", "nunique"),
          n_lines=("character_name", "size"),
          scene_text=("clean_text", " ".join)
      )
      .reset_index()
)

print("Scene-level dataset shape:", scene_df.shape)
scene_df.head()

Scene-level dataset shape: (165772, 6)


,movie_id,segment_id,characters_present,n_characters,n_lines,scene_text
0,10 Cloverfield Lane_1179933,0,"[Howard, Michelle]",2,30,michelle's eyes slowly open. she's back on the...
1,10 Cloverfield Lane_1179933,1,[Michelle],1,7,note all of the flashbacks are grainy and high...
2,10 Cloverfield Lane_1179933,2,"[Howard, Michelle]",2,3,my name's howard. michelle doesn't say anythin...
3,10 Cloverfield Lane_1179933,3,"[Howard, Michelle]",2,2,michelle enters the cramped living area. it's ...
4,10 Cloverfield Lane_1179933,4,"[Howard, Michelle]",2,24,howard lets the curtain go but stands right ou...


The initial scene aggregation using `(segment_id, scene_id)` resulted in an excessively granular representation, where most scenes contained only a single line of text.

To address this, we redefined the operational notion of a scene using only `segment_id`, effectively grouping micro-level fragments into broader narrative units.

This adjustment significantly reduced the number of scene units and produced a more meaningful representation of the screenplay structure.

The updated scene-level dataset exhibits:

- richer textual context per scene
- multiple characters co-occurring within the same unit
- improved suitability for interaction and network analysis

This refined definition is critical for downstream tasks such as:

- building character interaction graphs
- analyzing co-occurrence patterns
- modeling narrative dynamics across scenes

In [11]:
scene_df["n_characters"].value_counts().head(10)

n_characters
1     67255
2     52547
3     24222
4     11133
5      5185
6      2622
7      1245
8       626
9       344
10      217
Name: count, dtype: int64

In [12]:
scene_df["n_characters"].describe()

count    165772.000000
mean          2.143896
std           1.484543
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max          41.000000
Name: n_characters, dtype: float64

The distribution of characters per scene provides strong validation for the revised scene definition.

Approximately 60% of scenes contain two or more characters, indicating that the dataset captures a substantial number of interaction contexts.

This is particularly important for downstream tasks such as:

- building character interaction networks
- modeling co-occurrence relationships
- analyzing dialogue dynamics

The average number of characters per scene is approximately 2.14, with a median of 2, suggesting that most scenes involve direct interactions between a small number of characters.

Additionally, the presence of scenes with higher character counts (up to 40+) reflects complex narrative moments, such as group interactions or crowded settings.

Overall, these results confirm that the scene-level aggregation produces a meaningful and analytically useful representation of the screenplay structure.

## 4. Constructing character interaction edges

With the scene-level dataset in place, we can now construct the foundation of a character interaction network.

The key idea is that characters who appear in the same scene are considered to have interacted.

## Approach

For each scene, we generate all possible pairs of co-occurring characters.

These pairs represent potential interactions and will form the edges of a character network.

## Why this matters

This step transforms the dataset from a collection of text entries into a relational structure that can be used to:

- build graphs
- analyze character centrality
- detect key narrative roles

In [13]:
from itertools import combinations

edges = []

for _, row in scene_df.iterrows():
    chars = row["characters_present"]
    
    if len(chars) < 2:
        continue
    
    for pair in combinations(chars, 2):
        edges.append(pair)

edges_df = pd.DataFrame(edges, columns=["char_1", "char_2"])

print("Total edges:", len(edges_df))
edges_df.head()

Total edges: 385938


,char_1,char_2
0,Howard,Michelle
1,Howard,Michelle
2,Howard,Michelle
3,Howard,Michelle
4,Howard,Michelle


### Aggregating interaction strength

The raw edge list contains repeated pairs of characters, as they may co-occur in multiple scenes.

To build a meaningful interaction network, we aggregate these pairs and compute the number of times each pair appears.

This count represents the strength of the interaction between two characters.

## Why this matters

Weighted edges allow us to:

- identify strong relationships between characters
- distinguish central figures from peripheral ones
- construct more informative network representations

In [14]:
edges_weighted = (
    edges_df.groupby(["char_1", "char_2"])
            .size()
            .reset_index(name="weight")
            .sort_values("weight", ascending=False)
)

print("Unique edges:", edges_weighted.shape[0])
edges_weighted.head(10)

Unique edges: 117038


,char_1,char_2,weight
71873,Harry Potter,Hermione Granger,270
83016,John Brennan,John Look Alike,253
93593,M,R,224
61462,Forrest Gump,Forrest Junior,174
3172,Al,Ed,173
17336,Benjamin 1928 31,Benjamin 1935 37,171
17335,Benjamin 1928 31,Benjamin 1932 34,171
17337,Benjamin 1928 31,Benjamin Button,171
17362,Benjamin 1932 34,Benjamin 1935 37,171
17363,Benjamin 1932 34,Benjamin Button,171


The weighted interaction network reveals meaningful relationships between characters across the dataset.

High-frequency interactions between well-known character pairs (e.g., Harry Potter and Hermione Granger) confirm that the co-occurrence-based approach successfully captures narrative relationships.

However, the results also reveal inconsistencies in character naming, where variations of the same character appear as distinct entities (e.g., temporal versions or descriptive variations).

This highlights the need for a future character normalization step, which will be essential for improving the accuracy of network-based analyses.

Despite this limitation, the current representation already provides a strong foundation for interaction modeling.

In [15]:
char_centrality = (
    edges_weighted
    .assign(weight_total=lambda x: x["weight"])
    .groupby("char_1")
    .agg(
        total_interactions=("weight_total", "sum"),
        unique_connections=("char_2", "nunique")
    )
    .reset_index()
    .sort_values("total_interactions", ascending=False)
)

char_centrality.head(10)

,char_1,total_interactions,unique_connections
5442,Ed,1642,211
342,Al,1473,166
8449,Jack,1254,190
3117,Charlie,1158,250
412,Alex,1021,234
690,Andy,992,205
6909,George,940,231
6432,Frank,897,199
4348,David,861,209
4821,Doctor,797,506


The global centrality analysis provides a useful validation of the interaction pipeline, but it does not represent meaningful narrative importance.

Because the dataset contains multiple independent movies, aggregating all characters into a single network introduces ambiguity and noise, especially for common character names such as "Jack", "Alex", or "Doctor".

This highlights an important design consideration: character interaction analysis must be performed at the movie level, not globally.

The global analysis remains useful for validating the methodology and ensuring that interaction metrics behave as expected.

In subsequent steps, the same pipeline will be applied separately to each movie to obtain meaningful character-level insights.

## 5. Building character-level features

To analyze character roles and narrative importance, we need to aggregate the dataset at the character level.

Each row in this new dataset will represent a character within a specific movie.

## Why this matters

Character-level aggregation allows us to:

- identify protagonists and secondary characters
- measure narrative presence
- analyze participation across scenes
- build features for role classification

## Expected output

We expect to create a dataset containing, for each character:

- total number of lines
- number of scenes they appear in
- total word count
- average text length

In [16]:
character_df = (
    df.groupby(["movie_id", "character_name"])
      .agg(
          total_lines=("text", "size"),
          total_scenes=("scene_key", "nunique"),
          total_words=("word_count", "sum"),
          avg_words_per_line=("word_count", "mean"),
          avg_text_length=("text_length", "mean")
      )
      .reset_index()
      .sort_values("total_lines", ascending=False)
)

print("Character-level dataset shape:", character_df.shape)
character_df.head(10)

Character-level dataset shape: (32114, 7)


,movie_id,character_name,total_lines,total_scenes,total_words,avg_words_per_line,avg_text_length
26655,The Lost Son_0144286,Lombard,1017,1017,36547,35.936087,132.244838
15798,Molly s Game_4209788,Molly Bloom,978,978,19882,20.329243,109.314928
21564,Steve Jobs_2080374,Steve Capps,932,932,15526,16.658798,89.622318
21565,Steve Jobs_2080374,Steve Jobs,932,932,15526,16.658798,89.622318
21566,Steve Jobs_2080374,Steve Wozniak,932,932,15526,16.658798,89.622318
31926,Yes Man_1068680,Carl,899,899,9997,11.120133,60.033370
29331,The Wrestler_1125849,Randy The Ram Robinson,897,897,11241,12.531773,68.826087
12273,It s a Wonderful Life_0038650,George Bailey,895,895,14753,16.483799,89.785475
8378,Forrest Gump_0109830,Forrest Gump,863,863,13316,15.429896,82.374276
8379,Forrest Gump_0109830,Forrest Junior,858,858,13271,15.467366,82.560606


The character-level aggregation reveals meaningful patterns in narrative presence, with well-known protagonists appearing at the top of the ranking.

However, the results also expose an important structural property of the dataset.

In several cases, multiple characters from the same movie exhibit identical values for:

- total number of lines
- total number of scenes
- total word count

This suggests that the dataset assigns scene-level text to each character present in that scene, rather than isolating individual dialogue contributions.

As a result, the current metrics reflect **scene participation** rather than strictly individual dialogue volume.

## Implications

- The features are still useful for measuring narrative presence
- However, they should not be interpreted as exact measures of individual speaking activity
- This will influence how character importance is defined in later stages

This observation highlights the importance of understanding dataset construction before drawing conclusions from aggregated features.

In [17]:
character_df["importance_score"] = (
    character_df["total_lines"] * 0.5 +
    character_df["total_scenes"] * 0.3 +
    character_df["total_words"] * 0.0005
)

character_df = character_df.sort_values("importance_score", ascending=False)

character_df.head(10)

,movie_id,character_name,total_lines,total_scenes,total_words,avg_words_per_line,avg_text_length,importance_score
26655,The Lost Son_0144286,Lombard,1017,1017,36547,35.936087,132.244838,831.8735
15798,Molly s Game_4209788,Molly Bloom,978,978,19882,20.329243,109.314928,792.3410
21564,Steve Jobs_2080374,Steve Capps,932,932,15526,16.658798,89.622318,753.3630
21565,Steve Jobs_2080374,Steve Jobs,932,932,15526,16.658798,89.622318,753.3630
21566,Steve Jobs_2080374,Steve Wozniak,932,932,15526,16.658798,89.622318,753.3630
31926,Yes Man_1068680,Carl,899,899,9997,11.120133,60.033370,724.1985
12273,It s a Wonderful Life_0038650,George Bailey,895,895,14753,16.483799,89.785475,723.3765
29331,The Wrestler_1125849,Randy The Ram Robinson,897,897,11241,12.531773,68.826087,723.2205
8378,Forrest Gump_0109830,Forrest Gump,863,863,13316,15.429896,82.374276,697.0580
8379,Forrest Gump_0109830,Forrest Junior,858,858,13271,15.467366,82.560606,693.0355


The importance score provides a useful proxy for narrative presence by combining multiple character-level features.

While the score successfully highlights prominent characters, it also reflects structural limitations of the dataset, where multiple characters may share identical values due to shared scene-level text.

To obtain meaningful insights, the analysis must be performed within each movie rather than globally.

This leads to the next step: ranking characters within each movie to identify relative importance.

In [18]:
character_df["rank_in_movie"] = (
    character_df
    .groupby("movie_id")["importance_score"]
    .rank(method="dense", ascending=False)
)

character_df.sort_values(["movie_id", "rank_in_movie"]).head(20)

,movie_id,character_name,total_lines,total_scenes,total_words,avg_words_per_line,avg_text_length,importance_score,rank_in_movie
0,10 Cloverfield Lane_1179933,Howard,316,316,8641,27.344937,147.575949,257.1205,1.0
1,10 Cloverfield Lane_1179933,Michelle,261,261,6921,26.517241,143.245211,212.2605,2.0
12,10 Things I Hate About You_0147800,Kat Stratford,387,387,4187,10.819121,57.992248,311.6935,1.0
16,10 Things I Hate About You_0147800,Patrick Verona,324,324,3209,9.904321,53.941358,260.8045,2.0
2,10 Things I Hate About You_0147800,Bianca Stratford,237,237,2555,10.780591,59.561181,190.8775,3.0
4,10 Things I Hate About You_0147800,Cameron James,186,186,1844,9.913978,55.349462,149.7220,4.0
14,10 Things I Hate About You_0147800,Michael Eckman,146,146,1695,11.609589,65.232877,117.6475,5.0
11,10 Things I Hate About You_0147800,Joey Donner,123,123,1301,10.577236,57.528455,99.0505,6.0
13,10 Things I Hate About You_0147800,Mandella,82,82,982,11.975610,67.243902,66.0910,7.0
19,10 Things I Hate About You_0147800,Walter Stratford,77,77,698,9.064935,49.389610,61.9490,8.0


Ranking characters within each movie produces a much more meaningful representation of narrative importance.

Unlike the global ranking, which mixes characters from different movies, this approach evaluates each character relative to others within the same story.

The results show strong alignment with expected narrative roles, with main characters consistently ranked at the top, followed by secondary and minor roles.

This confirms that the importance score, despite its simplicity, is effective at capturing relative character significance when applied within a movie context.

This ranking provides a solid foundation for identifying:

- protagonists
- supporting characters
- minor roles

and will be used in later stages to enrich character-level analysis.

## 6. Saving processed datasets

After constructing scene-level and character-level analytical datasets, we store the results for reuse in later stages of the project.

This allows us to:

- avoid recomputation
- maintain a clear data pipeline
- ensure reproducibility

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

scene_df.to_parquet(processed_dir / "scene_dataset.parquet", index=False)
character_df.to_parquet(processed_dir / "character_dataset.parquet", index=False)

print("Processed datasets saved successfully.")

The processed datasets have been successfully saved in the `data/processed/` directory.

These datasets represent enriched and structured versions of the original data, ready for advanced analysis.

They will serve as the foundation for:

- interaction graph construction
- emotion analysis
- character evolution modeling

## Conclusion

In this notebook, we transformed the structured character-level dataset into analytical representations at both the scene and character levels.

Key achievements include:

- cleaning and normalizing textual data
- redefining scene boundaries for meaningful aggregation
- constructing scene-level interaction units
- building a character interaction network
- deriving character-level features and importance scores
- ranking characters within each movie

These steps establish a strong analytical foundation for the next phase of the project, where we will focus on:

- emotion analysis
- interaction graph modeling
- character role classification